# Paralelismo de memoria compatida

El paralelismo de memoria compartida es un paradigma de concurrencia que aplica cuando tenemos unidades lógicas de procesamiento que **comparten bancos de memoria**, i.e., se pueden sincronizar elementos de computación sin necesidad de realizar **pasos de mensajes** y basta con sincronizar sus **accesos de memoria**.

### Sistema multi-CPU y multinúcleo NUMA (*non-uniform memory access*)

<div style="text-align: center;">
<img src="Figs/Fig8.png" width="350">
</div>

## Procesos e hilos

<div style="text-align: center;">
<img src="Figs/Fig9.png" width="420">
</div>

* Un hilo es una imagen de un proceso: una instancia del programa con sus propios datos (memoria privada).
* Cada hilo puede seguir su propio flujo de control a través del programa.
* Los hilos pueden compartir datos con otros hilos, pero también contienen información privada a esos hilos.
* La comunicación inter-hilo ocurre mediante accesos al banco de memoria compartida.
* Existe un **hilo principal** cuya función es coordinar y sincronizar todos los hilos de un grupo.
* **Stack vs. heap**: en este modelo, el heap es compartido con todos los hilos. El stack es privado a cada hilo. Existe un espacio global donde las variables y objetos pueden vivir sin necesidad de uso de memoria dinámica.

## OpenMP

`OpenMP` (open specialisation for multi-processing) *no es un lenguaje de programación*.
* Trabaja en conjunto con otros lenguajes existentes de programación como C/C++.
* De momento, no es posible trabajar directamente OpenMP con Python. Esto require de bibliotecas externas como Cython o Numba.
* En Python, usualmente utilizamos multi-threading por debajo mediante diferentes bibliotecas pre-compiladas (e.g., numpy).

La biblioteca nos da acceso a una **application programming interface (API)**:
* Estas interfaces de programación proveen la forma mas portable para aplicaciones en paralelo.
* Contiene tres componentes principales:
    * Directivas de compilador
    * Rutinas que se invocan al momento de ejecución
    * Variables de ambiente

La biblioteca se puede utilizar en nuestro programa de **forma incremental**. Recordemos que el paradigma solo permite el uso de elementos que comparten memoria, por ende la biblioteca se utiliza solo para el paralelismo de nodos con arquitectura multi-procesador y/o muti-núcleos.

`OpenMP` esconde los llamados a la biblioteca que genera y manipula los hilos.
* Esto da poca flexibilidad en general, sin embargo, requiere de muy poca programación usualmente.
* De igual forma, la API procee estructuras para manipular el comportamiento de los hilos de forma directa (se desea usualmente evitar esto).
* **Peligro:** accesos de escritura a secciones de memoria compartidas puede dar lugar a condiciones de fallo de sincronización y/o corrupción de memoria.

### Modelo Fork-Join

<div style="text-align: center;">
<img src="Figs/Fig10.png" width="620">
</div>

* Hilos dinámicos.
* Paralelismo explícito.
* Declaraciones que ocurren mediante directivas al compilador.

#### Formato de las directivas

Una directiva es una línea especial de código fuente que sólo tiene significado para ciertos compiladores.

Las directivas se introducen **al inicio del scope de la región paralela** (cuando se realiza un *fork*):

`#pragma omp parallel`

* Esta directiva seguida de un scope, inicializa un ambiente de OpenMP.
* Ver `hello.cpp`: para la compilación, `g++ hello.cpp -o hello.x -fopenmp`
* Al ejecutar el binario, las hileras de caracteres impresas en `stdout` se pueden corromper y los hilos imprimen en desorden. ¿Porqué? Intente ejecutar el programa varias veces.
* La API ([ver la documentación](https://www.openmp.org/resources/refguides/)) permite invocar funciones que manipulan el comportamiento de los hilos.

### Huella de memoria

<div style="text-align: center;">
<img src="Figs/Fig11.png" width="420">
</div>

### Scopes de variables en ambientes paralelos

Todas las variables declaradas previas a un scope paralelo **siguen existiendo dentro del scope paralelo**:
* Por defecto, dichas variables son **compartidas** por todos los hilos.

Por otro lado, todas las variables declaradas dentro un scope paralelo son privadas a cada hilo por defecto.
* Se pueden declarar privadas o compartidas como parte de la directiva de compilador.
* Por ejemplo, los índices de un for loop en un ambiente paralelo son variables privadas.

La cláusula `firstprivate` inicializa instancias privadas de una variable u objeto con los contenidos de una variable compartida.

### Paralelismo de `for loop`s

En la gran mayoría de instancias de paralelismo en computación científica, nos interesa explotar el paralelismo para implementar for loops de manera concurrente.
* La idea es dividir el número de iteraciones entre el número total de hilos
    * Esto es muy fácil de implementar con OpenMP.
    * Da lugar a código muy fácil de leer.
    * Es la instancia de paralelismo más utilizada comúnmente.
* Ver `vector.cpp`: se puede controlar el número de hilos generando una variable de ambiente en `bash`, usando el comando `export OMP_NUM_THREADS=N`, donde `N` corresponde al número de hilos a utilizar.

### Práctica guiada

Acelerar la integración numérica de cuadratura homogénea usando `OpenMP`.